In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('credit_card_balance.csv', on_bad_lines='skip')

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
df.shape

(3840312, 23)

In [ ]:
df.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,1800.0,1800.0,0.000,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,2250.0,2250.0,60175.080,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,2250.0,2250.0,26926.425,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,11925.0,11925.0,224949.285,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,27000.0,27000.0,443044.395,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3840312 entries, 0 to 3840311
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   SK_ID_PREV                  int64  
 1   SK_ID_CURR                  int64  
 2   MONTHS_BALANCE              int64  
 3   AMT_BALANCE                 float64
 4   AMT_CREDIT_LIMIT_ACTUAL     int64  
 5   AMT_DRAWINGS_ATM_CURRENT    float64
 6   AMT_DRAWINGS_CURRENT        float64
 7   AMT_DRAWINGS_OTHER_CURRENT  float64
 8   AMT_DRAWINGS_POS_CURRENT    float64
 9   AMT_INST_MIN_REGULARITY     float64
 10  AMT_PAYMENT_CURRENT         float64
 11  AMT_PAYMENT_TOTAL_CURRENT   float64
 12  AMT_RECEIVABLE_PRINCIPAL    float64
 13  AMT_RECIVABLE               float64
 14  AMT_TOTAL_RECEIVABLE        float64
 15  CNT_DRAWINGS_ATM_CURRENT    float64
 16  CNT_DRAWINGS_CURRENT        int64  
 17  CNT_DRAWINGS_OTHER_CURRENT  float64
 18  CNT_DRAWINGS_POS_CURRENT    float64
 19  CNT_INSTALMENT_MATURE

Check **"SK_ID_CURR"** and **"SK_ID_PREV"**

In [ ]:
print("Rows:", len(df))

print("Unique customers :",
      df['SK_ID_CURR'].nunique())

print("Unique previous accounts :",
      df['SK_ID_PREV'].nunique())

print("SK_ID_PREV unique :",
      df['SK_ID_PREV'].is_unique)

Rows: 3840312
Unique customers : 103558
Unique previous accounts : 104307
SK_ID_PREV unique : False


MISSING VALUES

In [ ]:
missing_pct = (
    df.isna()
    .mean()
    .mul(100)
    .round(4)
    .sort_values(ascending=False)
)


missing_pct.head(10)

,0
AMT_PAYMENT_CURRENT,19.9981
CNT_DRAWINGS_POS_CURRENT,19.5249
AMT_DRAWINGS_ATM_CURRENT,19.5249
CNT_DRAWINGS_ATM_CURRENT,19.5249
AMT_DRAWINGS_POS_CURRENT,19.5249
AMT_DRAWINGS_OTHER_CURRENT,19.5249
CNT_DRAWINGS_OTHER_CURRENT,19.5249
CNT_INSTALMENT_MATURE_CUM,7.9482
AMT_INST_MIN_REGULARITY,7.9482
AMT_DRAWINGS_CURRENT,0.0000


CHECK CONTRACT STATUS

In [ ]:
df['NAME_CONTRACT_STATUS'].value_counts(dropna=False)

,count
NAME_CONTRACT_STATUS,
Active,3698436
Completed,128918
Signed,11058
Demand,1365
Sent proposal,513
Refused,17
Approved,5


In [ ]:
# ============================================================
# 1. ROW-LEVEL FEATURE ENGINEERING
# ============================================================

# Credit utilization:
# How much of the available credit limit is being used?
df['CARD_UTILIZATION'] = (
    df['AMT_BALANCE'] /
    df['AMT_CREDIT_LIMIT_ACTUAL'].replace(0, np.nan)
)

# Payment relative to balance:
# How much of the outstanding balance was paid?
df['PAYMENT_TO_BALANCE_RATIO'] = (
    df['AMT_PAYMENT_TOTAL_CURRENT'] /
    df['AMT_BALANCE'].replace(0, np.nan)
)

# Delinquency flag:
# 1 = payment was overdue, 0 = no overdue record
df['HAS_DPD'] = (
    df['SK_DPD'] > 0
).astype(int)

* **CARD_UTILIZATION** tells us about credit usage pressure.

* **PAYMENT_TO_BALANCE_RATIO** tells us about repayment behavior.

* **HAS_DPD** tells us whether the customer was delinquent in that month.

In [ ]:
# ============================================================
# 2. CARD-LEVEL AGGREGATION
# ============================================================

card_level = (
    df
    .groupby(['SK_ID_CURR', 'SK_ID_PREV'])
    .agg(

        # History
        CARD_MONTHS=('MONTHS_BALANCE', 'count'),

        # Credit exposure
        CARD_AVG_BALANCE=('AMT_BALANCE', 'mean'),
        CARD_MAX_BALANCE=('AMT_BALANCE', 'max'),
        CARD_MAX_CREDIT_LIMIT=(
            'AMT_CREDIT_LIMIT_ACTUAL', 'max'
        ),

        # Credit utilization
        CARD_AVG_UTILIZATION=(
            'CARD_UTILIZATION', 'mean'
        ),
        CARD_MAX_UTILIZATION=(
            'CARD_UTILIZATION', 'max'
        ),

        # Payment behavior
        CARD_TOTAL_PAYMENT=(
            'AMT_PAYMENT_TOTAL_CURRENT', 'sum'
        ),
        CARD_AVG_PAYMENT=(
            'AMT_PAYMENT_TOTAL_CURRENT', 'mean'
        ),

        # Payment relative to balance
        CARD_AVG_PAYMENT_BALANCE_RATIO=(
            'PAYMENT_TO_BALANCE_RATIO', 'mean'
        ),

        # Amount of card usage
        CARD_TOTAL_DRAWINGS=(
            'AMT_DRAWINGS_CURRENT', 'sum'
        ),
        CARD_TOTAL_ATM_DRAWINGS=(
            'AMT_DRAWINGS_ATM_CURRENT', 'sum'
        ),
        CARD_TOTAL_POS_DRAWINGS=(
            'AMT_DRAWINGS_POS_CURRENT', 'sum'
        ),
        CARD_TOTAL_OTHER_DRAWINGS=(
            'AMT_DRAWINGS_OTHER_CURRENT', 'sum'
        ),

        # Frequency of card usage
        CARD_TOTAL_DRAWING_COUNT=(
            'CNT_DRAWINGS_CURRENT', 'sum'
        ),
        CARD_TOTAL_ATM_DRAWING_COUNT=(
            'CNT_DRAWINGS_ATM_CURRENT', 'sum'
        ),
        CARD_TOTAL_POS_DRAWING_COUNT=(
            'CNT_DRAWINGS_POS_CURRENT', 'sum'
        ),
        CARD_TOTAL_OTHER_DRAWING_COUNT=(
            'CNT_DRAWINGS_OTHER_CURRENT', 'sum'
        ),

        # Installment history
        CARD_MAX_MATURED_INSTALLMENTS=(
            'CNT_INSTALMENT_MATURE_CUM', 'max'
        ),

        # Delinquency
        CARD_MAX_DPD=(
            'SK_DPD', 'max'
        ),
        CARD_AVG_DPD=(
            'SK_DPD', 'mean'
        ),
        CARD_DPD_MONTHS=(
            'HAS_DPD', 'sum'
        )
    )
    .reset_index()
)

In [ ]:
# ============================================================
# 3. Create card-level DPD rate
# ============================================================

card_level['CARD_DPD_RATE'] = (
    card_level['CARD_DPD_MONTHS'] /
    card_level['CARD_MONTHS'].replace(0, np.nan)
)

In [ ]:
card_level.sort_values(
    by='CARD_MONTHS',
    ascending=False
).head()

,SK_ID_CURR,SK_ID_PREV,CARD_MONTHS,CARD_AVG_BALANCE,CARD_MAX_BALANCE,CARD_MAX_CREDIT_LIMIT,CARD_AVG_UTILIZATION,CARD_MAX_UTILIZATION,CARD_TOTAL_PAYMENT,CARD_AVG_PAYMENT,CARD_AVG_PAYMENT_BALANCE_RATIO,CARD_TOTAL_DRAWINGS,CARD_TOTAL_ATM_DRAWINGS,CARD_TOTAL_POS_DRAWINGS,CARD_TOTAL_OTHER_DRAWINGS,CARD_TOTAL_DRAWING_COUNT,CARD_TOTAL_ATM_DRAWING_COUNT,CARD_TOTAL_POS_DRAWING_COUNT,CARD_TOTAL_OTHER_DRAWING_COUNT,CARD_MAX_MATURED_INSTALLMENTS,CARD_MAX_DPD,CARD_AVG_DPD,CARD_DPD_MONTHS,CARD_DPD_RATE
2,100013,2038692,96,18159.919219,161420.220,157500,0.115301,1.024890,654448.545,6817.172344,19.077035,571500.000,571500.0,0.000,0.0,23,23.0,0.0,0.0,22.0,1,0.010417,1,0.010417
104294,456225,2445843,96,117501.437344,197355.420,180000,0.652786,1.096419,925920.000,9645.000000,15.540941,681714.945,566550.0,115164.945,0.0,88,45.0,43.0,0.0,71.0,1,0.302083,29,0.302083
104283,456198,2287927,96,120398.047500,191455.110,180000,0.691958,1.063639,770553.000,8026.593750,0.901169,245475.000,245475.0,0.000,0.0,40,40.0,0.0,0.0,89.0,1,0.114583,11,0.114583
16887,157983,2603711,96,119115.064688,275343.255,270000,0.732392,1.094782,1451610.000,15120.937500,3.278609,846216.000,795600.0,50616.000,0.0,39,35.0,4.0,0.0,81.0,1,0.218750,21,0.218750
16913,158072,2720446,96,10699.494844,139297.050,135000,0.079256,1.031830,314167.500,3272.578125,41.350985,270000.000,270000.0,0.000,0.0,6,6.0,0.0,0.0,23.0,1,0.010417,1,0.010417


In [ ]:
# ============================================================
# 4. CUSTOMER-LEVEL AGGREGATION
# ============================================================

customer_features = (
    card_level
    .groupby('SK_ID_CURR')
    .agg(

        # Number of previous credit-card accounts
        CREDIT_CARD_COUNT=(
            'SK_ID_PREV', 'nunique'
        ),

        # Card history
        AVG_CARD_MONTHS=(
            'CARD_MONTHS', 'mean'
        ),
        MAX_CARD_MONTHS=(
            'CARD_MONTHS', 'max'
        ),

        # Credit exposure
        AVG_CARD_BALANCE=(
            'CARD_AVG_BALANCE', 'mean'
        ),
        MAX_CARD_BALANCE=(
            'CARD_MAX_BALANCE', 'max'
        ),
        MAX_CARD_CREDIT_LIMIT=(
            'CARD_MAX_CREDIT_LIMIT', 'max'
        ),

        # Credit utilization
        AVG_CARD_UTILIZATION=(
            'CARD_AVG_UTILIZATION', 'mean'
        ),
        MAX_CARD_UTILIZATION=(
            'CARD_MAX_UTILIZATION', 'max'
        ),

        # Payment behavior
        TOTAL_CARD_PAYMENTS=(
            'CARD_TOTAL_PAYMENT', 'sum'
        ),
        AVG_CARD_PAYMENT=(
            'CARD_AVG_PAYMENT', 'mean'
        ),
        AVG_PAYMENT_BALANCE_RATIO=(
            'CARD_AVG_PAYMENT_BALANCE_RATIO', 'mean'
        ),

        # Amount of card usage
        TOTAL_CARD_DRAWINGS=(
            'CARD_TOTAL_DRAWINGS', 'sum'
        ),
        TOTAL_ATM_DRAWINGS=(
            'CARD_TOTAL_ATM_DRAWINGS', 'sum'
        ),
        TOTAL_POS_DRAWINGS=(
            'CARD_TOTAL_POS_DRAWINGS', 'sum'
        ),
        TOTAL_OTHER_DRAWINGS=(
            'CARD_TOTAL_OTHER_DRAWINGS', 'sum'
        ),

        # Frequency of card usage
        TOTAL_DRAWING_COUNT=(
            'CARD_TOTAL_DRAWING_COUNT', 'sum'
        ),
        TOTAL_ATM_DRAWING_COUNT=(
            'CARD_TOTAL_ATM_DRAWING_COUNT', 'sum'
        ),
        TOTAL_POS_DRAWING_COUNT=(
            'CARD_TOTAL_POS_DRAWING_COUNT', 'sum'
        ),
        TOTAL_OTHER_DRAWING_COUNT=(
            'CARD_TOTAL_OTHER_DRAWING_COUNT', 'sum'
        ),

        # Installment history
        MAX_MATURED_INSTALLMENTS=(
            'CARD_MAX_MATURED_INSTALLMENTS', 'max'
        ),

        # Delinquency
        MAX_CARD_DPD=(
            'CARD_MAX_DPD', 'max'
        ),
        AVG_CARD_DPD=(
            'CARD_MAX_DPD', 'mean'
        ),
        TOTAL_CARD_DPD_MONTHS=(
            'CARD_DPD_MONTHS', 'sum'
        ),
        AVG_CARD_DPD_RATE=(
            'CARD_DPD_RATE', 'mean'
        ),
        MAX_CARD_DPD_RATE=(
            'CARD_DPD_RATE', 'max'
        )
    )
    .reset_index()
)

In [ ]:
# ============================================================
# 5. CONTRACT STATUS COUNTS
# ============================================================

status_counts = pd.crosstab(
    df['SK_ID_CURR'],
    df['NAME_CONTRACT_STATUS']
)

status_counts.columns = [
    f'CREDIT_CARD_STATUS_{col.upper().replace(" ", "_")}_COUNT'
    for col in status_counts.columns
]

In [ ]:
final_credit_card_features = (
    customer_features
    .merge(
        status_counts.reset_index(),
        on='SK_ID_CURR',
        how='left'
    )
    .fillna(0)
)

In [ ]:
final_credit_card_features.query("CREDIT_CARD_COUNT > 1").sort_values(
    by='CREDIT_CARD_COUNT',
    ascending=False
).head()

,SK_ID_CURR,CREDIT_CARD_COUNT,AVG_CARD_MONTHS,MAX_CARD_MONTHS,AVG_CARD_BALANCE,MAX_CARD_BALANCE,MAX_CARD_CREDIT_LIMIT,AVG_CARD_UTILIZATION,MAX_CARD_UTILIZATION,TOTAL_CARD_PAYMENTS,AVG_CARD_PAYMENT,AVG_PAYMENT_BALANCE_RATIO,TOTAL_CARD_DRAWINGS,TOTAL_ATM_DRAWINGS,TOTAL_POS_DRAWINGS,TOTAL_OTHER_DRAWINGS,TOTAL_DRAWING_COUNT,TOTAL_ATM_DRAWING_COUNT,TOTAL_POS_DRAWING_COUNT,TOTAL_OTHER_DRAWING_COUNT,MAX_MATURED_INSTALLMENTS,MAX_CARD_DPD,AVG_CARD_DPD,TOTAL_CARD_DPD_MONTHS,AVG_CARD_DPD_RATE,MAX_CARD_DPD_RATE,CREDIT_CARD_STATUS_ACTIVE_COUNT,CREDIT_CARD_STATUS_APPROVED_COUNT,CREDIT_CARD_STATUS_COMPLETED_COUNT,CREDIT_CARD_STATUS_DEMAND_COUNT,CREDIT_CARD_STATUS_REFUSED_COUNT,CREDIT_CARD_STATUS_SENT_PROPOSAL_COUNT,CREDIT_CARD_STATUS_SIGNED_COUNT
74145,355767,4,6.500000,12,0.000000,0.000,630000,0.000000,0.000000,0.000,0.000000,0.000000,0.000,0.0,0.00,0.0,0,0.0,0.0,0.0,0.0,0,0.000000,0,0.000000,0.000000,5,0,21,0,0,0,0
5713,120076,3,46.666667,96,34097.760156,235168.380,270000,0.136692,1.045193,2357481.285,31181.141094,6.147981,2216900.565,1240650.0,974865.24,0.0,237,28.0,209.0,0.0,35.0,0,0.000000,0,0.000000,0.000000,86,0,54,0,0,0,0
30848,206455,3,9.666667,24,0.000000,0.000,270000,0.000000,0.000000,0.000,0.000000,0.000000,0.000,0.0,0.00,0.0,0,0.0,0.0,0.0,0.0,0,0.000000,0,0.000000,0.000000,22,0,7,0,0,0,0
96387,431517,3,36.333333,90,153883.178250,534128.670,495000,0.386095,1.079048,677250.000,12738.750000,0.983077,861879.375,824400.0,34924.50,0.0,33,31.0,2.0,0.0,44.0,1,0.333333,2,0.007407,0.022222,64,0,45,0,0,0,0
25196,187294,3,34.333333,84,4148.635179,164248.155,450000,0.148925,0.912490,242793.540,1443.722198,32.495655,186075.000,173700.0,12375.00,0.0,10,7.0,3.0,0.0,12.0,0,0.000000,0,0.000000,0.000000,62,0,41,0,0,0,0


In [ ]:
# ============================================================
# 6. VALIDATION
# ============================================================

print("Shape:", final_credit_card_features.shape)

print(
    "Unique customers:",
    final_credit_card_features['SK_ID_CURR'].nunique()
)

print(
    "Duplicate customers:",
    final_credit_card_features['SK_ID_CURR'].duplicated().sum()
)

print("\nMissing values:")
display(
    final_credit_card_features
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head()
)

Shape: (103558, 33)
Unique customers: 103558
Duplicate customers: 0

Missing values:


,0
SK_ID_CURR,0
CREDIT_CARD_COUNT,0
AVG_CARD_MONTHS,0
MAX_CARD_MONTHS,0
AVG_CARD_BALANCE,0


### **CHANGING THE COL PREFIX_NAME (CC)**

| Current Name                             | New Name                        |
| ---------------------------------------- | ------------------------------- |
| `SK_ID_CURR`                             | `SK_ID_CURR`                    |
| `CREDIT_CARD_COUNT`                      | `CC_COUNT`                      |
| `AVG_CARD_MONTHS`                        | `CC_AVG_CARD_MONTHS`            |
| `MAX_CARD_MONTHS`                        | `CC_MAX_CARD_MONTHS`            |
| `AVG_CARD_BALANCE`                       | `CC_AVG_CARD_BALANCE`           |
| `MAX_CARD_BALANCE`                       | `CC_MAX_CARD_BALANCE`           |
| `MAX_CARD_CREDIT_LIMIT`                  | `CC_MAX_CARD_CREDIT_LIMIT`      |
| `AVG_CARD_UTILIZATION`                   | `CC_AVG_CARD_UTILIZATION`       |
| `MAX_CARD_UTILIZATION`                   | `CC_MAX_CARD_UTILIZATION`       |
| `TOTAL_CARD_PAYMENTS`                    | `CC_TOTAL_CARD_PAYMENTS`        |
| `AVG_CARD_PAYMENT`                       | `CC_AVG_CARD_PAYMENT`           |
| `AVG_PAYMENT_BALANCE_RATIO`              | `CC_AVG_PAYMENT_BALANCE_RATIO`  |
| `TOTAL_CARD_DRAWINGS`                    | `CC_TOTAL_CARD_DRAWINGS`        |
| `TOTAL_ATM_DRAWINGS`                     | `CC_TOTAL_ATM_DRAWINGS`         |
| `TOTAL_POS_DRAWINGS`                     | `CC_TOTAL_POS_DRAWINGS`         |
| `TOTAL_OTHER_DRAWINGS`                   | `CC_TOTAL_OTHER_DRAWINGS`       |
| `TOTAL_DRAWING_COUNT`                    | `CC_TOTAL_DRAWING_COUNT`        |
| `TOTAL_ATM_DRAWING_COUNT`                | `CC_TOTAL_ATM_DRAWING_COUNT`    |
| `TOTAL_POS_DRAWING_COUNT`                | `CC_TOTAL_POS_DRAWING_COUNT`    |
| `TOTAL_OTHER_DRAWING_COUNT`              | `CC_TOTAL_OTHER_DRAWING_COUNT`  |
| `MAX_MATURED_INSTALLMENTS`               | `CC_MAX_MATURED_INSTALLMENTS`   |
| `MAX_CARD_DPD`                           | `CC_MAX_CARD_DPD`               |
| `AVG_CARD_DPD`                           | `CC_AVG_CARD_DPD`               |
| `TOTAL_CARD_DPD_MONTHS`                  | `CC_TOTAL_CARD_DPD_MONTHS`      |
| `AVG_CARD_DPD_RATE`                      | `CC_AVG_CARD_DPD_RATE`          |
| `MAX_CARD_DPD_RATE`                      | `CC_MAX_CARD_DPD_RATE`          |
| `CREDIT_CARD_STATUS_ACTIVE_COUNT`        | `CC_STATUS_ACTIVE_COUNT`        |
| `CREDIT_CARD_STATUS_APPROVED_COUNT`      | `CC_STATUS_APPROVED_COUNT`      |
| `CREDIT_CARD_STATUS_COMPLETED_COUNT`     | `CC_STATUS_COMPLETED_COUNT`     |
| `CREDIT_CARD_STATUS_DEMAND_COUNT`        | `CC_STATUS_DEMAND_COUNT`        |
| `CREDIT_CARD_STATUS_REFUSED_COUNT`       | `CC_STATUS_REFUSED_COUNT`       |
| `CREDIT_CARD_STATUS_SENT_PROPOSAL_COUNT` | `CC_STATUS_SENT_PROPOSAL_COUNT` |
| `CREDIT_CARD_STATUS_SIGNED_COUNT`        | `CC_STATUS_SIGNED_COUNT`        |


In [ ]:
final_credit_card_features = final_credit_card_features.rename(
    columns=lambda col: (
        col.replace("CREDIT_CARD", "CC", 1)
        if "CREDIT_CARD" in col
        else col if col == "SK_ID_CURR"
        else f"CC_{col}"
    )
)

In [ ]:
final_credit_card_features.head()

,SK_ID_CURR,CC_COUNT,CC_AVG_CARD_MONTHS,CC_MAX_CARD_MONTHS,CC_AVG_CARD_BALANCE,CC_MAX_CARD_BALANCE,CC_MAX_CARD_CREDIT_LIMIT,CC_AVG_CARD_UTILIZATION,CC_MAX_CARD_UTILIZATION,CC_TOTAL_CARD_PAYMENTS,CC_AVG_CARD_PAYMENT,CC_AVG_PAYMENT_BALANCE_RATIO,CC_TOTAL_CARD_DRAWINGS,CC_TOTAL_ATM_DRAWINGS,CC_TOTAL_POS_DRAWINGS,CC_TOTAL_OTHER_DRAWINGS,CC_TOTAL_DRAWING_COUNT,CC_TOTAL_ATM_DRAWING_COUNT,CC_TOTAL_POS_DRAWING_COUNT,CC_TOTAL_OTHER_DRAWING_COUNT,CC_MAX_MATURED_INSTALLMENTS,CC_MAX_CARD_DPD,CC_AVG_CARD_DPD,CC_TOTAL_CARD_DPD_MONTHS,CC_AVG_CARD_DPD_RATE,CC_MAX_CARD_DPD_RATE,CC_STATUS_ACTIVE_COUNT,CC_STATUS_APPROVED_COUNT,CC_STATUS_COMPLETED_COUNT,CC_STATUS_DEMAND_COUNT,CC_STATUS_REFUSED_COUNT,CC_STATUS_SENT_PROPOSAL_COUNT,CC_STATUS_SIGNED_COUNT
0,100006,1,6.0,6,0.000000,0.00,270000,0.000000,0.00000,0.000,0.000000,0.000000,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0.000000,0.000000,6,0,0,0,0,0,0
1,100011,1,74.0,74,54482.111149,189000.00,180000,0.302678,1.05000,334485.000,4520.067568,2.167712,180000.0,180000.0,0.0,0.0,4,4.0,0.0,0.0,33.0,0,0.0,0,0.000000,0.000000,74,0,0,0,0,0,0
2,100013,1,96.0,96,18159.919219,161420.22,157500,0.115301,1.02489,654448.545,6817.172344,19.077035,571500.0,571500.0,0.0,0.0,23,23.0,0.0,0.0,22.0,1,1.0,1,0.010417,0.010417,96,0,0,0,0,0,0
3,100021,1,17.0,17,0.000000,0.00,675000,0.000000,0.00000,0.000,0.000000,0.000000,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0.000000,0.000000,7,0,10,0,0,0,0
4,100023,1,8.0,8,0.000000,0.00,225000,0.000000,0.00000,0.000,0.000000,0.000000,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0.000000,0.000000,8,0,0,0,0,0,0


SAVE IT

In [ ]:
final_credit_card_features.to_csv(
    'credit_card_aggregated.csv',
    index=False
)